# Gunshot Detection & Trimming Pipeline
## Clean Production Pipeline - WAV Audio Processing
Input: Edge-Collected Gunshot Audio (WAV files)
Output: Trimmed gunshot clips with empty files removed

In [1]:
%pip install -q numpy pandas librosa opencv-python tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations
import logging
from dataclasses import dataclass
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
LOGGER = logging.getLogger('gunshot_pipeline')

C:\Users\Bhupendra\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class PipelineConfig:
    dataset_root: Path
    output_root: Path
    sample_rate: int = 22050
    frame_length: int = 2048
    hop_length: int = 256
    zscore_threshold: float = 3.5
    min_event_gap_s: float = 0.15
    low_band_hz: float = 150.0
    high_band_hz: float = 3500.0
    pre_event_pad_s: float = 0.01
    post_event_pad_s: float = 0.02

def detect_audio_candidates(audio_path, config):
    try:
        y, sr = librosa.load(str(audio_path), sr=config.sample_rate, mono=True)
        if len(y) < config.frame_length:
            return []
        stft = np.abs(librosa.stft(y, n_fft=config.frame_length, hop_length=config.hop_length))
        freqs = librosa.fft_frequencies(sr=sr, n_fft=config.frame_length)
        band_mask = (freqs >= config.low_band_hz) & (freqs <= config.high_band_hz)
        band_energy = stft[band_mask].sum(axis=0)
        band_mean, band_std = band_energy.mean(), band_energy.std()
        band_zscore = (band_energy - band_mean) / (band_std + 1e-6)
        spectral_flux = np.sqrt(np.sum(np.diff(stft, axis=1) ** 2, axis=0))
        sf_mean, sf_std = spectral_flux.mean(), spectral_flux.std()
        sf_zscore = (spectral_flux - sf_mean) / (sf_std + 1e-6)
        combined_score = 0.6 * np.pad(band_zscore, (0, 1), mode='edge') + 0.4 * sf_zscore
        peaks = np.where(combined_score > config.zscore_threshold)[0]
        if len(peaks) == 0:
            return []
        events = []
        current_start = peaks[0]
        current_end = peaks[0]
        for peak in peaks[1:]:
            frame_gap_s = (peak - current_end) * config.hop_length / sr
            if frame_gap_s > config.min_event_gap_s:
                start_time = current_start * config.hop_length / sr
                end_time = current_end * config.hop_length / sr
                events.append({'start': start_time, 'end': end_time})
                current_start = peak
            current_end = peak
        start_time = current_start * config.hop_length / sr
        end_time = current_end * config.hop_length / sr
        events.append({'start': start_time, 'end': end_time})
        return events
    except Exception as e:
        LOGGER.warning(f"Detection failed: {e}")
        return []

def find_exact_gunshot_boundaries(audio_path, event, config):
    try:
        y, sr = librosa.load(str(audio_path), sr=config.sample_rate, mono=True)
        stft = np.abs(librosa.stft(y, n_fft=config.frame_length, hop_length=config.hop_length))
        freqs = librosa.fft_frequencies(sr=sr, n_fft=config.frame_length)
        band_mask = (freqs >= config.low_band_hz) & (freqs <= config.high_band_hz)
        band_energy = stft[band_mask].sum(axis=0)
        band_energy_norm = band_energy / (np.max(band_energy) + 1e-6)
        threshold = 0.15
        start_frame_approx = int(event['start'] * sr / config.hop_length)
        end_frame_approx = int(event['end'] * sr / config.hop_length)
        if end_frame_approx >= len(band_energy_norm):
            end_frame_approx = len(band_energy_norm) - 1
        start_frame = start_frame_approx
        for i in range(start_frame_approx, max(0, start_frame_approx - 50), -1):
            if band_energy_norm[i] < threshold:
                start_frame = i + 1
                break
        end_frame = end_frame_approx
        for i in range(end_frame_approx, min(len(band_energy_norm), end_frame_approx + 50)):
            if band_energy_norm[i] < threshold:
                end_frame = i - 1
                break
        start_time = max(0, start_frame * config.hop_length / sr - config.pre_event_pad_s)
        end_time = min(len(y) / sr, end_frame * config.hop_length / sr + config.post_event_pad_s)
        return {'start': start_time, 'end': end_time}
    except:
        event['start'] = max(0, event['start'] - config.pre_event_pad_s)
        event['end'] = event['end'] + config.post_event_pad_s
        return event

def run_pipeline_on_wavs(config):
    config.output_root.mkdir(parents=True, exist_ok=True)
    clips_dir = config.output_root / "clips"
    clips_dir.mkdir(exist_ok=True)
    audio_files = sorted(config.dataset_root.rglob('*.wav'))
    report_rows = []
    for audio_file in tqdm(audio_files, desc="Processing audio files"):
        category = audio_file.parent.name
        category_dir = clips_dir / category
        category_dir.mkdir(exist_ok=True)
        events = detect_audio_candidates(audio_file, config)
        segments_exported = 0
        for i, event in enumerate(events):
            precise_event = find_exact_gunshot_boundaries(audio_file, event, config)
            y, sr = librosa.load(str(audio_file), sr=config.sample_rate, mono=True)
            start_sample = int(precise_event['start'] * sr)
            end_sample = int(precise_event['end'] * sr)
            if start_sample < 0:
                start_sample = 0
            if end_sample > len(y):
                end_sample = len(y)
            if end_sample - start_sample > sr // 10:
                segment = y[start_sample:end_sample]
                output_path = category_dir / f"{audio_file.stem}_seg{i:02d}.wav"
                librosa.output.write_wav(str(output_path), segment, sr=sr)
                segments_exported += 1
        report_rows.append({
            'audio_file': audio_file.name,
            'category': category,
            'events_detected': len(events),
            'segments_exported': segments_exported,
            'status': 'success'
        })
    report_df = pd.DataFrame(report_rows)
    report_path = config.output_root / "report.csv"
    report_df.to_csv(report_path, index=False)
    LOGGER.info(f"Report saved: {report_path}")
    return report_df

In [4]:
print("\n" + "="*80)
print("GUNSHOT DETECTION & TRIMMING PIPELINE")
print("="*80)
PRODUCTION_CONFIG = PipelineConfig(
    dataset_root=Path(r'C:\Users\Bhupendra\Data-Cleaner\Data\edge-collected-gunshot-audio\edge-collected-gunshot-audio'),
    output_root=Path(r'C:\Users\Bhupendra\Data-Cleaner\Data\FINAL_GUNSHOT_TRIMMED'),
    sample_rate=22050,
    frame_length=2048,
    hop_length=256,
    zscore_threshold=3.5,
    min_event_gap_s=0.15,
    low_band_hz=150.0,
    high_band_hz=3500.0,
    pre_event_pad_s=0.01,
    post_event_pad_s=0.02,
)
PRODUCTION_CONFIG.output_root.mkdir(parents=True, exist_ok=True)
print(f"\n📂 INPUT: {PRODUCTION_CONFIG.dataset_root}")
print(f"📂 OUTPUT: {PRODUCTION_CONFIG.output_root}")
print(f"\n⚙️ Parameters:")
print(f"   Sample Rate: 22050 Hz")
print(f"   Freq Range: 150-3500 Hz")
print(f"   Detection Threshold: 3.5")
print(f"   Padding: 10ms before + 20ms after")
print(f"\n🔄 Ready to process...")


GUNSHOT DETECTION & TRIMMING PIPELINE

📂 INPUT: C:\Users\Bhupendra\Data-Cleaner\Data\edge-collected-gunshot-audio\edge-collected-gunshot-audio
📂 OUTPUT: C:\Users\Bhupendra\Data-Cleaner\Data\FINAL_GUNSHOT_TRIMMED

⚙️ Parameters:
   Sample Rate: 22050 Hz
   Freq Range: 150-3500 Hz
   Detection Threshold: 3.5
   Padding: 10ms before + 20ms after

🔄 Ready to process...


In [5]:
print("\n" + "="*80)
print("STEP 1: DETECT & TRIM GUNSHOTS")
print("="*80)

import soundfile as sf
config = PRODUCTION_CONFIG
config.output_root.mkdir(parents=True, exist_ok=True)
clips_dir = config.output_root / "clips"
clips_dir.mkdir(exist_ok=True)

# Clear old output
import shutil
if clips_dir.exists():
    shutil.rmtree(clips_dir)
    clips_dir.mkdir(exist_ok=True)

# FIX: Redefine detection function with shape fix
def detect_audio_candidates_fixed(audio_path, config):
    try:
        y, sr = librosa.load(str(audio_path), sr=config.sample_rate, mono=True)
        if len(y) < config.frame_length:
            return []
        stft = np.abs(librosa.stft(y, n_fft=config.frame_length, hop_length=config.hop_length))
        freqs = librosa.fft_frequencies(sr=sr, n_fft=config.frame_length)
        band_mask = (freqs >= config.low_band_hz) & (freqs <= config.high_band_hz)
        band_energy = stft[band_mask].sum(axis=0)
        band_mean, band_std = band_energy.mean(), band_energy.std()
        band_zscore = (band_energy - band_mean) / (band_std + 1e-6)
        spectral_flux = np.sqrt(np.sum(np.diff(stft, axis=1) ** 2, axis=0))
        sf_mean, sf_std = spectral_flux.mean(), spectral_flux.std()
        sf_zscore = (spectral_flux - sf_mean) / (sf_std + 1e-6)
        
        # FIX: Ensure same length - spectral_flux is one shorter due to diff
        min_len = min(len(band_zscore), len(sf_zscore))
        band_zscore = band_zscore[:min_len]
        sf_zscore = sf_zscore[:min_len]
        
        combined_score = 0.6 * band_zscore + 0.4 * sf_zscore
        peaks = np.where(combined_score > config.zscore_threshold)[0]
        
        if len(peaks) == 0:
            return []
        events = []
        current_start = peaks[0]
        current_end = peaks[0]
        for peak in peaks[1:]:
            frame_gap_s = (peak - current_end) * config.hop_length / sr
            if frame_gap_s > config.min_event_gap_s:
                start_time = current_start * config.hop_length / sr
                end_time = current_end * config.hop_length / sr
                events.append({'start': start_time, 'end': end_time})
                current_start = peak
            current_end = peak
        start_time = current_start * config.hop_length / sr
        end_time = current_end * config.hop_length / sr
        events.append({'start': start_time, 'end': end_time})
        return events
    except Exception as e:
        LOGGER.warning(f"Detection failed: {e}")
        return []

audio_files = sorted(config.dataset_root.rglob('*.wav'))
report_rows = []

print(f"Found {len(audio_files)} WAV files to process...")
print(f"Detection threshold: {config.zscore_threshold}")
print(f"Frequency range: {config.low_band_hz}-{config.high_band_hz} Hz\n")

total_events = 0
total_length_filtered = 0

for audio_file in tqdm(audio_files, desc="Processing audio files"):
    category = audio_file.parent.name
    category_dir = clips_dir / category
    category_dir.mkdir(exist_ok=True)
    
    events = detect_audio_candidates_fixed(audio_file, config)
    segments_exported = 0
    length_filtered = 0
    
    for i, event in enumerate(events):
        precise_event = find_exact_gunshot_boundaries(audio_file, event, config)
        y, sr = librosa.load(str(audio_file), sr=config.sample_rate, mono=True)
        
        start_sample = int(precise_event['start'] * sr)
        end_sample = int(precise_event['end'] * sr)
        
        if start_sample < 0:
            start_sample = 0
        if end_sample > len(y):
            end_sample = len(y)
        
        segment_length = end_sample - start_sample
        if segment_length > sr // 10:
            segment = y[start_sample:end_sample]
            output_path = category_dir / f"{audio_file.stem}_seg{i:02d}.wav"
            sf.write(str(output_path), segment, sr)
            segments_exported += 1
        else:
            length_filtered += 1
    
    total_events += len(events)
    total_length_filtered += length_filtered
    
    report_rows.append({
        'audio_file': audio_file.name,
        'category': category,
        'events_detected': len(events),
        'segments_exported': segments_exported,
        'status': 'success'
    })

final_report = pd.DataFrame(report_rows)
report_path = config.output_root / "report.csv"
final_report.to_csv(report_path, index=False)
LOGGER.info(f"Report saved: {report_path}")

print(f"\n✅ Trimming completed!")
print(f"   Files processed: {len(final_report)}")
print(f"   Events detected: {total_events}")
print(f"   Filtered (too short): {total_length_filtered}")
print(f"   Segments exported: {final_report['segments_exported'].sum()}")


STEP 1: DETECT & TRIM GUNSHOTS
Found 2148 WAV files to process...
Detection threshold: 3.5
Frequency range: 150.0-3500.0 Hz



Processing audio files: 100%|██████████| 2148/2148 [01:17<00:00, 27.73it/s]
[INFO] Report saved: C:\Users\Bhupendra\Data-Cleaner\Data\FINAL_GUNSHOT_TRIMMED\report.csv



✅ Trimming completed!
   Files processed: 2148
   Events detected: 3428
   Filtered (too short): 768
   Segments exported: 2660


In [6]:
print("\n" + "="*80)
print("STEP 2: VALIDATE & REMOVE EMPTY FILES")
print("="*80)
clips_dir = PRODUCTION_CONFIG.output_root / "clips"
print(f"\nValidating files in: {clips_dir}\n")

def is_valid_gunshot(wav_path):
    try:
        y, sr = librosa.load(str(wav_path), sr=22050, mono=True)
        if len(y) == 0 or np.max(np.abs(y)) < 0.01:
            return False
        stft = np.abs(librosa.stft(y, n_fft=2048, hop_length=256))
        freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
        band_mask = (freqs >= 150) & (freqs <= 3500)
        band_energy = stft[band_mask].sum(axis=0)
        if np.mean(band_energy) < 0.1:
            return False
        peak_frames = np.where(band_energy > np.mean(band_energy) * 2)[0]
        return len(peak_frames) >= 3
    except:
        return False

valid_count = 0
removed_count = 0
for wav_file in clips_dir.rglob("*.wav"):
    if not is_valid_gunshot(wav_file):
        try:
            wav_file.unlink()
            removed_count += 1
        except:
            pass
    else:
        valid_count += 1
print(f"✅ Valid gunshot clips: {valid_count}")
print(f"❌ Empty/silent files removed: {removed_count}")


STEP 2: VALIDATE & REMOVE EMPTY FILES

Validating files in: C:\Users\Bhupendra\Data-Cleaner\Data\FINAL_GUNSHOT_TRIMMED\clips

✅ Valid gunshot clips: 730
❌ Empty/silent files removed: 1930


In [7]:
print("\n" + "="*80)
print("FINAL OUTPUT SUMMARY")
print("="*80)
category_stats = {}
for category_dir in sorted(clips_dir.iterdir()):
    if category_dir.is_dir():
        count = len(list(category_dir.glob("*.wav")))
        category_stats[category_dir.name] = count
print("\n📊 Gunshot Clips by Firearm Type:\n")
total_final = 0
for category in sorted(category_stats.keys()):
    count = category_stats[category]
    print(f"   ✅ {category:35s} : {count:4d} clips")
    total_final += count
print(f"\n   {'─'*50}")
print(f"   TOTAL VALID GUNSHOT CLIPS: {total_final:4d}")
print(f"\n📈 STATISTICS:")
print(f"   Original trimmed files:     {final_report['segments_exported'].sum()}")
print(f"   Empty/silent files removed:  {removed_count}")
print(f"   Final clean output:          {total_final}")
if final_report['segments_exported'].sum() > 0:
    pct = (total_final/final_report['segments_exported'].sum())*100
    print(f"   Percentage kept:             {pct:.1f}%")
print("\n" + "="*80)
print("📁 OUTPUT LOCATION:")
print("="*80)
print(f"\n{PRODUCTION_CONFIG.output_root}")
print(f"\n✨ All files contain ONLY actual gunshot sound!")
print(f"   No empty, silent, or invalid clips in output.")
print("\n" + "="*80)


FINAL OUTPUT SUMMARY

📊 Gunshot Clips by Firearm Type:

   ✅ 38s&ws_dot38_caliber                :  189 clips
   ✅ glock_17_9mm_caliber                :  227 clips
   ✅ remington_870_12_gauge              :  114 clips
   ✅ ruger_ar_556_dot223_caliber         :  200 clips

   ──────────────────────────────────────────────────
   TOTAL VALID GUNSHOT CLIPS:  730

📈 STATISTICS:
   Original trimmed files:     2660
   Empty/silent files removed:  1930
   Final clean output:          730
   Percentage kept:             27.4%

📁 OUTPUT LOCATION:

C:\Users\Bhupendra\Data-Cleaner\Data\FINAL_GUNSHOT_TRIMMED

✨ All files contain ONLY actual gunshot sound!
   No empty, silent, or invalid clips in output.

